# 35 — Education Parsing
**Goal:** Extract degree, institution, field, and graduation year.

Education is one of the most regular blocks on a resume: a degree name, an institution, a field, and a year, usually on one or two lines. Its regularity makes it a perfect regex target — no ML required. This chapter builds a degree vocabulary, an extractor that pulls the four fields, and an alias table that normalizes institution names.

**Why it matters for resumes / ATS:** education is a hard filter in many postings ("MS in CS required"). Structured extraction lets an ATS answer "does this candidate hold a Master's in a relevant field?" reliably — and normalizing "IIT Bombay" to "Indian Institute of Technology Bombay" keeps those checks consistent across thousands of differently-phrased resumes.

## 1. Education Patterns

The extractor's vocabulary is a list of degree strings covering the common variants: bachelor's (`B.Tech`, `B.E.`, `B.S.`, `B.Sc`, `B.A.`), master's (`M.S.`, `M.Tech`, `M.B.A.`, `MBA`), doctorates (`PhD`, `Ph.D.`, `Doctorate`), generic words (`Bachelors`, `Masters`), and even pre-university markers (`10th`, `12th`, `SSC`, `HSC`) common in some regions.

**What the code does:** defines `DEGREES` and prints `Known degree patterns: 27` — the full count of patterns the matcher will try against each line.

**Why it matters:** coverage here defines recall. Generic forms like "Masters" catch informal resumes that "M.S." misses, while regional markers (`10th`/`12th`) matter for markets where school-leaving certificates are listed — a reminder that a production vocabulary should be tuned to the candidate population.

In [ ]:
DEGREES = [
    "B.Tech", "B.E.", "B.S.", "B.Sc", "B.A.", "B.Com", "B.B.A.",
    "M.Tech", "M.E.", "M.S.", "M.Sc", "M.A.", "M.Com", "M.B.A.", "MBA",
    "PhD", "Ph.D.", "Doctorate",
    "Bachelors", "Masters", "Bachelor", "Master",
    "10th", "12th", "Higher Secondary", "SSC", "HSC",
]
print(f"Known degree patterns: {len(DEGREES)}")

## 2. Education Extractor

`extract_education()` applies the standard recipe: split into lines, scan each line for any degree pattern, then pull the surrounding fields with three small regexes — institution after "at"/"from", a 19xx/20xx year, and a field after "in".

**What the code does:** for each matched line it assembles a dict with `degree`, `institution`, `field`, `year`, and a `confidence` of `"high"` when both institution and year were found, else `"medium"`. It stops at the first degree per line.

**Honest failure modes:** as written, the degree check uses `r"\\b"` (escaped backslash) instead of `r"\b"`, so no line matches and the cell prints nothing. With the boundary fixed, two of the three test entries come out — `B.Tech` / Computer Science / 2019 (medium: no "at"/"from" preposition) and `PhD` / NLP / no year (medium) — while "M.S. in Data Science from Stanford University, 2021" is *skipped entirely*, because the trailing `\b` after the period in `M.S.` requires a word character next, and a space is not one. That is the classic `\b`-ends-on-punctuation trap.

**Try it:** the `(?:at|from)` alternation explains why "from Stanford University" yields an institution but "IIT Bombay" after a comma does not.

In [ ]:
import re

def extract_education(text):
    """Extract education entries from resume text."""
    entries = []
    lines = text.split("\n")
    for i, line in enumerate(lines):
        line = line.strip()
        # Check for degree name
        for degree in DEGREES:
            if re.search(r"\\b" + re.escape(degree) + r"\\b", line, re.IGNORECASE):
                # Try to extract institution (look for "at", "from", or next line)
                inst_match = re.search(r"(?:at|from)\s+([A-Z][A-Za-z\s.]+)", line)
                institution = inst_match.group(1) if inst_match else ""
                
                # Year
                year_match = re.search(r"\b(19\d{2}|20\d{2})\b", line)
                year = year_match.group(1) if year_match else None
                
                # Field
                field_match = re.search(r"in\s+([A-Za-z\s]+)", line)
                field = field_match.group(1).strip() if field_match else ""
                
                entries.append({
                    "degree": degree, "institution": institution,
                    "field": field, "year": year,
                    "confidence": "high" if (institution and year) else "medium"
                })
                break
    return entries

text = """B.Tech in Computer Science, IIT Bombay, 2019
M.S. in Data Science from Stanford University, 2021
PhD in NLP, MIT (ongoing)"""
for e in extract_education(text):
    print(f"  {e['degree']:8s} in {e['field']:20s} @ {e['institution']:20s} ({e['year']}) [{e['confidence']}]")

## 3. Institution Normalization

Institutions are spelled a dozen ways ("IIT Bombay" vs "Indian Institute of Technology Bombay", "MIT" vs "Massachusetts Institute of Technology"). An alias map collapses them onto full canonical names — the same idea as Ch. 34's skill taxonomy, applied to universities.

**What the code does:** `UNIVERSITY_ALIASES` maps 11 shorthand keys (lowercased) to full names; `normalize_institution()` lowercases the input and looks it up, returning the input unchanged when there is no alias.

**Verified on the sample:** `"IIT Bombay"` → `Indian Institute of Technology Bombay`, `"Stanford"` → `Stanford University`, `"MIT"` → `Massachusetts Institute of Technology` — and `"NIT Trichy"` → `NIT Trichy` unchanged, because the map only contains the generic key `"nit"`, not specific campuses.

**Try it:** that `"nit"` key is effectively a wildcard — it will rewrite *any* "NIT ..." entry to the generic name. Decide whether you want that behavior before deploying.

In [ ]:
UNIVERSITY_ALIASES = {
    "iit bombay": "Indian Institute of Technology Bombay",
    "iit delhi": "Indian Institute of Technology Delhi",
    "iit madras": "Indian Institute of Technology Madras",
    "iit kanpur": "Indian Institute of Technology Kanpur",
    "iit kharagpur": "Indian Institute of Technology Kharagpur",
    "nit": "National Institute of Technology",
    "stanford": "Stanford University",
    "mit": "Massachusetts Institute of Technology",
    "harvard": "Harvard University",
    "oxford": "University of Oxford",
    "cambridge": "University of Cambridge",
}

def normalize_institution(name):
    key = name.strip().lower()
    return UNIVERSITY_ALIASES.get(key, name)

for inst in ["IIT Bombay", "Stanford", "MIT", "NIT Trichy"]:
    print(f"  '{inst}' -> '{normalize_institution(inst)}'")

## Summary: Regex extracts structured education data. Institution aliases normalize names.

**Education is the most regular resume block — and the most reliably regex-parseable one.**

A 27-pattern degree vocabulary, three field regexes, and an 11-entry alias table are enough to extract degree, institution, field, and year with a confidence signal. The extraction dicts are already schema-shaped: they map 1:1 onto the `Education` model in Ch. 39, with `confidence` ready to flow into `ExtractedField`. And the `\b`-escaping lesson carries forward — verify a regex result before trusting it.

Next, Ch. 36 applies the same line-state-machine approach to the messier, bullet-heavy experience section.